## Boston Bautista
## Group 3
## Group Assignment Task 2

Question: Which industries show the most consistent profitability and which experience the highest volatility?

In [24]:
from pyspark.sql import SparkSession
from math import sqrt

def split_csv(line):
    return line.split(",")

def safe_float(x):
    try:
        return float(x)
    except:
        return None

In [25]:
# Create Spark session connected to localhost cluster
spark = (
    SparkSession.builder.appName("Stock Market Analysis")
    # .master("spark://localhost:7077")
    .getOrCreate()
)

# Get Spark context from session
sc = spark.sparkContext

# Set log level to reduce verbosity
sc.setLogLevel("WARN")

print("✅ Connected to Spark cluster!")
print(f"Spark Version: {spark.version}")
print(f"Master: {sc.master}")
print(f"App ID: {sc.applicationId}")

✅ Connected to Spark cluster!
Spark Version: 4.0.1
Master: local[*]
App ID: local-1764700596048


25/12/02 10:36:36 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/12/02 10:36:36 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [26]:
import argparse

parser = argparse.ArgumentParser()
parser.add_argument(
    "--is-local",
    type=str,
    default="true",
    help="Whether running in local mode",
)

args, unknown = parser.parse_known_args()
is_local = args.is_local.lower() == "true"
if is_local:
    prefix = "../data/processed/merged"
else:
    prefix = "gs://msds-694-cohort-14-3/data"

print(f"Is local environment: {is_local}")

num_csv_path = f"{prefix}/num_2020.csv"
pre_csv_path = f"{prefix}/pre_2020.csv"
sub_csv_path = f"{prefix}/sub_2020.csv"
tag_csv_path = f"{prefix}/tag_2020.csv"

print(f"Num CSV path: {num_csv_path}")
print(f"Pre CSV path: {pre_csv_path}")
print(f"Sub CSV path: {sub_csv_path}")
print(f"Tag CSV path: {tag_csv_path}")

num_rdd = sc.textFile(num_csv_path)
pre_rdd = sc.textFile(pre_csv_path)
sub_rdd = sc.textFile(sub_csv_path)
tag_rdd = sc.textFile(tag_csv_path)

# print size of each RDD
print(f"Num RDD size: {num_rdd.count()}")
print(f"Pre RDD size: {pre_rdd.count()}")
print(f"Sub RDD size: {sub_rdd.count()}")
print(f"Tag RDD size: {tag_rdd.count()}")

Is local environment: True
Num CSV path: ../data/processed/merged/num_2020.csv
Pre CSV path: ../data/processed/merged/pre_2020.csv
Sub CSV path: ../data/processed/merged/sub_2020.csv
Tag CSV path: ../data/processed/merged/tag_2020.csv


Num RDD size: 11493263
Pre RDD size: 2746310
Sub RDD size: 24940
Tag RDD size: 298803


In [27]:
num_header_line = num_rdd.first()
sub_header_line = sub_rdd.first()
num_header = num_header_line.split(",")
sub_header = sub_header_line.split(",")

num_body_rdd = (num_rdd.filter(lambda l: l != num_header_line).map(split_csv).filter(lambda r: len(r) == len(num_header)))

sub_body_rdd = (sub_rdd.filter(lambda l: l != sub_header_line).map(split_csv).filter(lambda r: len(r) == len(sub_header)))

In [28]:
n_adsh  = num_header.index("adsh")
n_tag   = num_header.index("tag")
n_uom   = num_header.index("uom")
n_seg   = num_header.index("segments")
n_coreg = num_header.index("coreg")
n_val   = num_header.index("value")
s_adsh  = sub_header.index("adsh")
s_sic   = sub_header.index("sic")

wanted_tags = {"NetIncomeLoss", "Assets"}

num_filtered_rdd = (num_body_rdd.filter(lambda r: r[n_tag] in wanted_tags and r[n_uom] == "USD" and r[n_seg] == "" and r[n_coreg] == "" ))

In [29]:
def pairs(row):
    adsh = row[n_adsh]
    tag  = row[n_tag]
    val  = safe_float(row[n_val])
    if val is None:
        return None

    if tag == "NetIncomeLoss":
        return (adsh, (val, None))
    else:  # Assets
        return (adsh, (None, val))

num_pairs_rdd = num_filtered_rdd.map(pairs).filter(lambda x: x is not None)

def combine(a, b):
    net_income = a[0] if a[0] is not None else b[0]
    assets     = a[1] if a[1] is not None else b[1]
    return (net_income, assets)

filings_rdd = num_pairs_rdd.reduceByKey(combine)

In [30]:
def sub_to_pair(row):
    adsh = row[s_adsh]
    sic_raw = row[s_sic]
    sic_val = safe_float(sic_raw)
    if sic_val is None:
        return None
    sic_code = str(int(sic_val))
    return (adsh, sic_code)

sub_pairs_rdd = sub_body_rdd.map(sub_to_pair).filter(lambda x: x is not None)

In [31]:
joined_rdd = filings_rdd.join(sub_pairs_rdd)

def to_sic_roa(record):
    adsh, data = record
    (net_income, assets), sic_code = data

    if net_income is None or assets is None:
        return None

    ni  = safe_float(net_income)
    at = safe_float(assets)
    if ni is None or at is None or at <= 0:
        return None

    roa = ni / at
    return (sic_code, (roa, 1))

sic_roa_rdd = joined_rdd.map(to_sic_roa).filter(lambda x: x is not None)

In [32]:
def roa(a, b):
    sum_roa = a[0] + b[0]
    count   = a[1] + b[1]
    return (sum_roa, count)

sic_sums = sic_roa_rdd.reduceByKey(roa)

def finalize_avg(item):
    sic, (sum_roa, n) = item
    if n == 0:
        return (sic, None, n)
    avg_roa = sum_roa / n
    return (sic, avg_roa, n)

sic_avg_roa_rdd = sic_sums.map(finalize_avg)

In [33]:
top = (sic_avg_roa_rdd.filter(lambda x: x[1] is not None).sortBy(lambda x: x[1], ascending=False).take(10))

print("Top 10 industries by average ROA")
for row in top:
    print(row)

Top 10 industries by average ROA
('7363', 9.307148109032335, 39)
('3841', 9.053561294748267, 233)
('3679', 1.596932582974086, 40)
('6795', 0.5311063853219786, 8)
('5094', 0.448868210513924, 2)
('3564', 0.1384842130047258, 8)
('5080', 0.11122812180591468, 24)
('3577', 0.1077042450846116, 25)
('8900', 0.10356306747177042, 4)
('6519', 0.07742471602488603, 8)


In [34]:
sc.stop()